# 📝 Datathon 2026 — Phase 2: MCQ Answers (10 Questions, 20 Points)

Each question is worth **2 points**. All answers are computed directly from the data.

| Q | Answer | Confidence |
|---|--------|------------|
| Q1 | _(computed below)_ | |
| Q2 | _(computed below)_ | |
| Q3 | _(computed below)_ | |
| Q4 | _(computed below)_ | |
| Q5 | _(computed below)_ | |
| Q6 | _(computed below)_ | |
| Q7 | _(computed below)_ | |
| Q8 | _(computed below)_ | |
| Q9 | _(computed below)_ | |
| Q10 | _(computed below)_ | |

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'

# Load required data
products    = pd.read_csv(DATA_DIR + 'products.csv')
customers   = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
geography   = pd.read_csv(DATA_DIR + 'geography.csv')
orders      = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')
payments    = pd.read_csv(DATA_DIR + 'payments.csv')
returns     = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
reviews     = pd.read_csv(DATA_DIR + 'reviews.csv', parse_dates=['review_date'])
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])
sales_train = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])

print('✅ Data loaded')

---
## Q1. Trung vị inter-order gap của khách hàng mua > 1 lần

**Câu hỏi**: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?

- A) 30 ngày
- B) 90 ngày
- C) 144 ngày
- D) 365 ngày

In [ ]:
# Q1: Median inter-order gap for customers with > 1 order
# Step 1: Keep only customers with more than 1 order
order_counts = orders.groupby('customer_id')['order_id'].count()
repeat_customers = order_counts[order_counts > 1].index
print(f"Total customers: {orders['customer_id'].nunique():,}")
print(f"Customers with > 1 order: {len(repeat_customers):,}")

# Step 2: Sort by customer and date, compute gaps
repeat_orders = orders[orders['customer_id'].isin(repeat_customers)].copy()
repeat_orders = repeat_orders.sort_values(['customer_id', 'order_date'])
repeat_orders['prev_order_date'] = repeat_orders.groupby('customer_id')['order_date'].shift(1)
repeat_orders['gap_days'] = (repeat_orders['order_date'] - repeat_orders['prev_order_date']).dt.days

# Step 3: Drop NaN gaps (first order per customer) and compute median
gaps = repeat_orders['gap_days'].dropna()
median_gap = gaps.median()

print(f"\nTotal inter-order gaps computed: {len(gaps):,}")
print(f"Mean gap: {gaps.mean():.1f} days")
print(f"Median gap: {median_gap:.1f} days")
print(f"25th percentile: {gaps.quantile(0.25):.1f} days")
print(f"75th percentile: {gaps.quantile(0.75):.1f} days")

print(f"\n✅ Q1 ANSWER: ~{median_gap:.0f} days → ", end='')
if median_gap < 60:
    print("A) 30 ngày")
elif median_gap < 120:
    print("B) 90 ngày")
elif median_gap < 250:
    print("C) 144 ngày")
else:
    print("D) 365 ngày")

---
## Q2. Segment có gross margin trung bình cao nhất

**Câu hỏi**: Phân khúc sản phẩm (`segment`) nào có tỷ suất lợi nhuận gộp trung bình cao nhất? $(price - cogs) / price$

- A) Premium
- B) Performance
- C) Activewear
- D) Standard

In [ ]:
# Q2: Segment with highest average gross margin
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

gm_by_segment = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print("Gross Margin by Segment:")
print(gm_by_segment.to_string())
print(f"\nProduct count per segment:")
print(products['segment'].value_counts().to_string())

best_segment = gm_by_segment.idxmax()
print(f"\n✅ Q2 ANSWER: {best_segment} (GM = {gm_by_segment.max():.4f})")

---
## Q3. Lý do trả hàng nhiều nhất của danh mục Streetwear

**Câu hỏi**: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear, lý do trả hàng nào xuất hiện nhiều nhất?

- A) defective
- B) wrong_size
- C) changed_mind
- D) not_as_described

In [ ]:
# Q3: Most common return reason for Streetwear products
# Join returns with products on product_id
returns_products = returns.merge(products[['product_id', 'category']], on='product_id', how='left')

# Filter Streetwear
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']
print(f"Total returns: {len(returns):,}")
print(f"Streetwear returns: {len(streetwear_returns):,}")

reason_counts = streetwear_returns['return_reason'].value_counts()
print(f"\nReturn reasons for Streetwear:")
print(reason_counts.to_string())

top_reason = reason_counts.idxmax()
print(f"\n✅ Q3 ANSWER: {top_reason} ({reason_counts.max():,} returns)")

---
## Q4. Traffic source có bounce_rate trung bình thấp nhất

**Câu hỏi**: Nguồn truy cập nào có tỷ lệ thoát trung bình thấp nhất?

- A) organic_search
- B) paid_search
- C) email_campaign
- D) social_media

In [ ]:
# Q4: Traffic source with lowest average bounce rate
bounce_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print("Average Bounce Rate by Traffic Source:")
print(bounce_by_source.to_string())
print(f"\nRow counts per source:")
print(web_traffic['traffic_source'].value_counts().to_string())

lowest = bounce_by_source.idxmin()
print(f"\n✅ Q4 ANSWER: {lowest} (avg bounce_rate = {bounce_by_source.min():.6f})")

---
## Q5. % dòng order_items có áp dụng promo

**Câu hỏi**: Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (promo_id không null)?

- A) 12%
- B) 25%
- C) 39%
- D) 54%

In [ ]:
# Q5: Percentage of order_items with promo applied
total_rows = len(order_items)
promo_rows = order_items['promo_id'].notna().sum()
pct_promo = (promo_rows / total_rows) * 100

print(f"Total order_items rows: {total_rows:,}")
print(f"Rows with promo_id (not null): {promo_rows:,}")
print(f"Percentage: {pct_promo:.2f}%")

# Also check promo_id_2 if relevant
if 'promo_id_2' in order_items.columns:
    promo2_rows = order_items['promo_id_2'].notna().sum()
    print(f"Rows with promo_id_2 (not null): {promo2_rows:,} ({promo2_rows/total_rows*100:.2f}%)")

print(f"\n✅ Q5 ANSWER: ~{pct_promo:.0f}% → ", end='')
if pct_promo < 18:
    print("A) 12%")
elif pct_promo < 32:
    print("B) 25%")
elif pct_promo < 46:
    print("C) 39%")
else:
    print("D) 54%")

---
## Q6. Age group có trung bình đơn hàng/khách cao nhất

**Câu hỏi**: Nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng đơn / số KH trong nhóm, chỉ xét age_group không null)

- A) 55+
- B) 25–34
- C) 35–44
- D) 45–54

In [ ]:
# Q6: Age group with highest average orders per customer
# Merge orders with customers to get age_group
orders_cust = orders.merge(customers[['customer_id', 'age_group']], on='customer_id', how='left')

# Filter out null age_group
orders_cust = orders_cust[orders_cust['age_group'].notna()]

# Count orders per age_group
orders_per_group = orders_cust.groupby('age_group')['order_id'].count()
print("Total orders per age_group:")
print(orders_per_group.to_string())

# Count unique customers per age_group
customers_valid = customers[customers['age_group'].notna()]
cust_per_group = customers_valid.groupby('age_group')['customer_id'].nunique()
print("\nCustomers per age_group:")
print(cust_per_group.to_string())

# Average orders per customer
avg_orders = (orders_per_group / cust_per_group).sort_values(ascending=False)
print("\nAvg orders per customer by age_group:")
print(avg_orders.to_string())

best_age = avg_orders.idxmax()
print(f"\n✅ Q6 ANSWER: {best_age} (avg {avg_orders.max():.2f} orders/customer)")

---
## Q7. Region có tổng doanh thu cao nhất

**Câu hỏi**: Vùng (`region`) nào tạo ra tổng doanh thu cao nhất?

- A) West
- B) Central
- C) East
- D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

**Cách tiếp cận**: Tính doanh thu từ order_items (unit_price × quantity − discount), join orders→geography để lấy region.

In [ ]:
# Q7: Region with highest total revenue
# Compute revenue per order_item
oi = order_items.copy()
oi['revenue'] = oi['unit_price'] * oi['quantity'] - oi['discount_amount']

# Join with orders to get zip
oi_orders = oi.merge(orders[['order_id', 'zip']], on='order_id', how='left')

# Join with geography to get region
oi_geo = oi_orders.merge(geography[['zip', 'region']].drop_duplicates(), on='zip', how='left')

# Revenue by region
rev_by_region = oi_geo.groupby('region')['revenue'].sum().sort_values(ascending=False)
print("Total Revenue by Region:")
for region, rev in rev_by_region.items():
    print(f"  {region}: {rev:,.0f} ({rev/rev_by_region.sum()*100:.1f}%)")

# Check if they're approximately equal
rev_vals = rev_by_region.values
cv = np.std(rev_vals) / np.mean(rev_vals)
print(f"\nCoefficient of variation: {cv:.4f}")
if cv < 0.05:
    print("→ Revenues are approximately equal")

top_region = rev_by_region.idxmax()
print(f"\n✅ Q7 ANSWER: {top_region} (revenue = {rev_by_region.max():,.0f})")

---
## Q8. Payment method nhiều nhất trong đơn cancelled

**Câu hỏi**: Trong đơn hàng bị cancelled, phương thức thanh toán nào được dùng nhiều nhất?

- A) credit_card
- B) cod
- C) paypal
- D) bank_transfer

In [ ]:
# Q8: Most common payment method in cancelled orders
cancelled = orders[orders['order_status'] == 'cancelled']
print(f"Total orders: {len(orders):,}")
print(f"Cancelled orders: {len(cancelled):,} ({len(cancelled)/len(orders)*100:.2f}%)")

payment_counts = cancelled['payment_method'].value_counts()
print(f"\nPayment methods in cancelled orders:")
print(payment_counts.to_string())

top_payment = payment_counts.idxmax()
print(f"\n✅ Q8 ANSWER: {top_payment} ({payment_counts.max():,} cancelled orders)")

---
## Q9. Size có tỷ lệ trả hàng cao nhất

**Câu hỏi**: Trong 4 size (S, M, L, XL), size nào có tỷ lệ trả hàng cao nhất? (số bản ghi returns / số dòng order_items, join với products theo product_id)

- A) S
- B) M
- C) L
- D) XL

In [ ]:
# Q9: Size with highest return rate (returns rows / order_items rows per size)
# Join order_items with products to get size
oi_prod = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_by_size = oi_prod.groupby('size')['order_id'].count()
print("Order items by size:")
print(oi_by_size.to_string())

# Join returns with products to get size
ret_prod = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_by_size = ret_prod.groupby('size')['return_id'].count()
print("\nReturns by size:")
print(ret_by_size.to_string())

# Return rate = returns / order_items per size
return_rate = (ret_by_size / oi_by_size).sort_values(ascending=False)
print("\nReturn rate by size:")
for size, rate in return_rate.items():
    print(f"  {size}: {rate:.4f} ({rate*100:.2f}%)")

top_size = return_rate.idxmax()
print(f"\n✅ Q9 ANSWER: {top_size} (return rate = {return_rate.max():.4f})")

---
## Q10. Kế hoạch trả góp có giá trị TB/đơn cao nhất

**Câu hỏi**: Kế hoạch trả góp nào có giá trị thanh toán trung bình cao nhất?

- A) 1 kỳ (trả một lần)
- B) 3 kỳ
- C) 6 kỳ
- D) 12 kỳ

In [ ]:
# Q10: Installment plan with highest average payment value
avg_payment_by_install = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print("Average payment value by installment plan:")
print(avg_payment_by_install.to_string())

print("\nOrder count per installment plan:")
print(payments['installments'].value_counts().sort_index().to_string())

top_install = avg_payment_by_install.idxmax()
print(f"\n✅ Q10 ANSWER: {top_install} installments (avg = {avg_payment_by_install.max():,.2f})")

---
## 📋 FINAL ANSWER SUMMARY

Run the cell below to get the consolidated answer key.

In [ ]:
print('=' * 60)
print('📋 DATATHON 2026 — MCQ ANSWER KEY')
print('=' * 60)

# Re-compute all answers in one place
answers = {}

# Q1
order_counts = orders.groupby('customer_id')['order_id'].count()
repeat_cust = order_counts[order_counts > 1].index
ro = orders[orders['customer_id'].isin(repeat_cust)].sort_values(['customer_id', 'order_date'])
ro['gap'] = ro.groupby('customer_id')['order_date'].diff().dt.days
q1_val = ro['gap'].dropna().median()
q1_map = {30: 'A', 90: 'B', 144: 'C', 365: 'D'}
q1_ans = min(q1_map.keys(), key=lambda x: abs(x - q1_val))
answers['Q1'] = f"{q1_map[q1_ans]}) {q1_ans} ngày  [computed: {q1_val:.1f}]"

# Q2
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
gm = products.groupby('segment')['gross_margin'].mean()
q2_seg = gm.idxmax()
q2_map = {'Premium': 'A', 'Performance': 'B', 'Activewear': 'C', 'Standard': 'D'}
answers['Q2'] = f"{q2_map.get(q2_seg, '?')}) {q2_seg}  [GM = {gm.max():.4f}]"

# Q3
rp = returns.merge(products[['product_id', 'category']], on='product_id', how='left')
q3_reason = rp[rp['category'] == 'Streetwear']['return_reason'].value_counts().idxmax()
q3_map = {'defective': 'A', 'wrong_size': 'B', 'changed_mind': 'C', 'not_as_described': 'D'}
answers['Q3'] = f"{q3_map.get(q3_reason, '?')}) {q3_reason}"

# Q4
br = web_traffic.groupby('traffic_source')['bounce_rate'].mean()
q4_src = br.idxmin()
q4_map = {'organic_search': 'A', 'paid_search': 'B', 'email_campaign': 'C', 'social_media': 'D'}
answers['Q4'] = f"{q4_map.get(q4_src, '?')}) {q4_src}  [BR = {br.min():.6f}]"

# Q5
q5_pct = order_items['promo_id'].notna().sum() / len(order_items) * 100
q5_map = {12: 'A', 25: 'B', 39: 'C', 54: 'D'}
q5_ans = min(q5_map.keys(), key=lambda x: abs(x - q5_pct))
answers['Q5'] = f"{q5_map[q5_ans]}) ~{q5_ans}%  [computed: {q5_pct:.2f}%]"

# Q6
oc = orders.merge(customers[['customer_id', 'age_group']], on='customer_id', how='left')
oc = oc[oc['age_group'].notna()]
opg = oc.groupby('age_group')['order_id'].count()
cv = customers[customers['age_group'].notna()].groupby('age_group')['customer_id'].nunique()
avg_o = (opg / cv).sort_values(ascending=False)
q6_age = avg_o.idxmax()
q6_map_keys = {'55+': 'A', '25-34': 'B', '25–34': 'B', '35-44': 'C', '35–44': 'C', '45-54': 'D', '45–54': 'D'}
answers['Q6'] = f"{q6_map_keys.get(q6_age, '?')}) {q6_age}  [avg = {avg_o.max():.2f}]"

# Q7
oi2 = order_items.copy()
oi2['revenue'] = oi2['unit_price'] * oi2['quantity'] - oi2['discount_amount']
oi_o = oi2.merge(orders[['order_id', 'zip']], on='order_id', how='left')
oi_g = oi_o.merge(geography[['zip', 'region']].drop_duplicates(), on='zip', how='left')
rbr = oi_g.groupby('region')['revenue'].sum()
q7_region = rbr.idxmax()
q7_cv = np.std(rbr.values) / np.mean(rbr.values)
q7_map = {'West': 'A', 'Central': 'B', 'East': 'C'}
if q7_cv < 0.05:
    answers['Q7'] = f"D) Cả ba vùng xấp xỉ bằng nhau  [CV = {q7_cv:.4f}]"
else:
    answers['Q7'] = f"{q7_map.get(q7_region, '?')}) {q7_region}  [rev = {rbr.max():,.0f}, CV = {q7_cv:.4f}]"

# Q8
canc = orders[orders['order_status'] == 'cancelled']
q8_pm = canc['payment_method'].value_counts().idxmax()
q8_map = {'credit_card': 'A', 'cod': 'B', 'paypal': 'C', 'bank_transfer': 'D'}
answers['Q8'] = f"{q8_map.get(q8_pm, '?')}) {q8_pm}"

# Q9
oi_p = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_sz = oi_p.groupby('size')['order_id'].count()
rt_p = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
rt_sz = rt_p.groupby('size')['return_id'].count()
rr = (rt_sz / oi_sz).sort_values(ascending=False)
q9_size = rr.idxmax()
q9_map = {'S': 'A', 'M': 'B', 'L': 'C', 'XL': 'D'}
answers['Q9'] = f"{q9_map.get(q9_size, '?')}) {q9_size}  [rate = {rr.max():.4f}]"

# Q10
avg_pv = payments.groupby('installments')['payment_value'].mean()
q10_inst = avg_pv.idxmax()
q10_map = {1: 'A', 3: 'B', 6: 'C', 12: 'D'}
answers['Q10'] = f"{q10_map.get(q10_inst, '?')}) {q10_inst} kỳ  [avg = {avg_pv.max():,.2f}]"

# Print summary
for q, a in answers.items():
    print(f"  {q}: {a}")
print('\n' + '=' * 60)